# PARC2026 00: Environment Check
Google Driveをマウントし、固定ブランチを取得してA100・Disk・Git Commitを確認します。

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
!ls -la /content/drive/MyDrive/PARC2026/40_experiments/datasets/ 2>/dev/null || echo "NOT FOUND"
!df -h /content/drive/MyDrive | tail -1

NOT FOUND
drive            15G   13G  2.5G  84% /content/drive


In [ ]:
from pathlib import Path
REPO = Path('/content/Physical_ai')
BRANCH = 'main'
if not REPO.exists():
    !git clone --branch {BRANCH} https://github.com/yu37330/Physical_ai.git {REPO}
else:
    !git -C {REPO} fetch origin {BRANCH}
    !git -C {REPO} checkout {BRANCH}
    !git -C {REPO} pull --ff-only origin {BRANCH}
%cd /content/Physical_ai


In [ ]:
!bash training/openvla_oft_a100/scripts/colab_preflight.sh

In [7]:
%cd /content
!rm -rf Physical_ai openvla-oft work && mkdir -p work
!git clone -b fix/colab-dlimp-tensorflow-pin https://github.com/yu37330/Physical_ai.git
%cd /content/Physical_ai

/content
Cloning into 'Physical_ai'...
remote: Enumerating objects: 1162, done.
remote: Counting objects: 100% (494/494), done.
remote: Compressing objects: 100% (342/342), done.
remote: Total 1162 (delta 196), reused 228 (delta 112), pack-reused 668 (from 3)
Receiving objects: 100% (1162/1162), 444.04 KiB | 7.66 MiB/s, done.
Resolving deltas: 100% (550/550), done.
/content/Physical_ai


In [8]:
!REQUIRE_DRIVE=0 REQUIRE_A100=0 bash training/openvla_oft_a100/scripts/colab_setup.sh 2>&1 | tee /content/work/setup.log


=== Runtime ===
ERROR: nvidia-smi was not found. Reconnect the notebook to a Colab GPU runtime.
ERROR: torch.cuda.is_available() is False. The current kernel is not a GPU runtime.
{
  "python": "3.12.13",
  "python_executable": "/usr/bin/python3",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "torch": "2.11.0+cpu",
  "cuda_available": false,
  "gpu_name": null,
  "vram_gib": null,
  "nvidia_smi": null
}


In [9]:
!bash training/openvla_oft_a100/scripts/colab_action_parity.sh 2>&1 | tee /content/work/parity.log

Checkpoint not found: /content/work/models/openvla_oft_plus_base
Run colab_setup.sh first.


In [7]:
!cd /content/Physical_ai && git pull
!bash training/openvla_oft_a100/scripts/colab_build_selection.sh

remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 27 (delta 18), reused 26 (delta 17), pack-reused 0 (from 0)
Unpacking objects: 100% (27/27), 6.14 KiB | 1.02 MiB/s, done.
From https://github.com/yu37330/Physical_ai
   cb638ee..bc4511a  fix/colab-dlimp-tensorflow-pin -> origin/fix/colab-dlimp-tensorflow-pin
Updating cb638ee..bc4511a
Fast-forward
 .github/workflows/ci.yml                           |  1 +
 scripts/capture_action_chunks.py                   |  7 ++
 src/data/inspect_lerobot_metadata.py               |  2 +-
 tests/test_data_modules_importable.py              | 46 +++++++++++++
 tests/test_dataset_pipeline.py                     | 15 ++--
 .../scripts/colab_action_parity.sh                 |  5 ++
 .../scripts/colab_build_selection.sh               | 79 ++++++++++++++++++++++
 .../scripts/colab_dataset_prepare.sh               |  9 ++-
 8 files changed, 154 insertions(+), 10 del

In [8]:
!ls -d /content/work/models/openvla_oft_plus_base 2>/dev/null || echo "CHECKPOINT MISSING"
!df -h /content | tail -1

/content/work/models/openvla_oft_plus_base
overlay         113G   72G   41G  64% /


In [9]:
!cd /content/Physical_ai && DATASET_PROFILE=mini bash training/openvla_oft_a100/scripts/colab_dataset_prepare.sh 2>&1 | tee /content/work/dataset.log


=== Mini selection (2 train / 1 validation) ===
{
  "total": 3,
  "by_split": {
    "train": 2,
    "validation": 1
  }
}
open the middle drawer of the cabinet

=== Download plan ===
{
  "episode_count": 3,
  "file_count": 9
}

=== Download episodes to /content/work/source/mini ===
Fetching 9 files: 100%|██████████| 9/9 [00:02<00:00,  3.44it/s]
{
  "repo_id": "Sylvest/libero_plus_lerobot",
  "resolved_revision": "22c57433fef692b5b9ecc0795344daac7fa867a5",
  "episode_count": 3,
  "include_videos": true,
  "local_dir": "/content/work/source/mini",
  "plan": "/content/drive/MyDrive/PARC2026/40_experiments/datasets/mini_download_plan.json"
}

=== LeRobot -> RLDS, parity, and OpenVLA batch compatibility ===
Ignoring tensorflow: markers 'python_version < "3.12"' don't match your environment
Ignoring tensorflow-datasets: markers 'python_version < "3.12"' don't match your environment
Ignoring tensorflow-metadata: markers 'python_version < "3.12"' don't match your environment
Ignoring protobuf

In [10]:
!cd /content/Physical_ai && git pull
!cd /content/Physical_ai && python -m pip install -q "av>=13,<16"
!cd /content/Physical_ai && DATASET_PROFILE=mini bash training/openvla_oft_a100/scripts/colab_dataset_prepare.sh 2>&1 | tail -40

remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 13 (delta 8), reused 13 (delta 8), pack-reused 0 (from 0)
Unpacking objects: 100% (13/13), 3.62 KiB | 1.21 MiB/s, done.
From https://github.com/yu37330/Physical_ai
   bc4511a..f165879  fix/colab-dlimp-tensorflow-pin -> origin/fix/colab-dlimp-tensorflow-pin
Updating bc4511a..f165879
Fast-forward
 .github/workflows/ci.yml                        |  2 +-
 src/data/convert_selected_lerobot_to_rlds.py    | 53 ++++++++++----
 tests/test_video_decoding.py                    | 91 +++++++++++++++++++++++++
 training/openvla_oft_a100/requirements-data.txt |  3 +
 4 files changed, 136 insertions(+), 13 deletions(-)
 create mode 100644 tests/test_video_decoding.py
2026-08-04 12:22:51.958441: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already 

In [11]:
!cd /content/Physical_ai && git pull
!cd /content/Physical_ai && DATASET_PROFILE=mini bash training/openvla_oft_a100/scripts/colab_dataset_prepare.sh 2>&1 | tail -40

remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 15 (delta 10), reused 15 (delta 10), pack-reused 0 (from 0)
Unpacking objects: 100% (15/15), 4.04 KiB | 1.01 MiB/s, done.
From https://github.com/yu37330/Physical_ai
   f165879..addb723  fix/colab-dlimp-tensorflow-pin -> origin/fix/colab-dlimp-tensorflow-pin
Updating f165879..addb723
Fast-forward
 .github/workflows/ci.yml                           |  1 +
 src/data/rlds_contract.py                          | 52 +++++++++++--
 tests/test_rlds_batch_contract.py                  | 91 ++++++++++++++++++++++
 tests/test_rlds_contract.py                        | 13 ++--
 .../scripts/validate_rlds_batch_transform.py       | 17 +++-
 5 files changed, 159 insertions(+), 15 deletions(-)
 create mode 100644 tests/test_rlds_batch_contract.py
08/04 [12:30:06] INFO     | >> [*] Loading existing dataset    data_utils.py:199
                          statisti

In [6]:
import json, time, subprocess
from pathlib import Path

R = Path('/content/drive/MyDrive/PARC2026/40_experiments/datasets/mini_e2e_v001')
for name in ('rlds_source_parity.json', 'openvla_rlds_compatibility.json'):
    d = json.loads((R / name).read_text())
    print(f'{name:36} -> {d["status"]}')

conv = json.loads((R / 'rlds_conversion_report.json').read_text())
print('episodes:', conv['episode_counts'], 'frames:', conv['frame_counts'])

print('--- 800 Episode の見積もり材料 ---')
!du -sh /content/work/source/mini /content/work/rlds/mini
!df -h /content | tail -1

rlds_source_parity.json              -> pass
openvla_rlds_compatibility.json      -> pass
episodes: {'train': 2, 'validation': 1} frames: {'train': 0, 'validation': 0}
--- 800 Episode の見積もり材料 ---
du: cannot access '/content/work/source/mini': No such file or directory
du: cannot access '/content/work/rlds/mini': No such file or directory
overlay         108G   20G   88G  19% /


In [13]:
!cd /content/Physical_ai && git pull
!cd /content/Physical_ai && DATASET_PROFILE=full MANIFEST_FILE=/content/drive/MyDrive/PARC2026/40_experiments/datasets/dataset_manifest.json bash training/openvla_oft_a100/scripts/colab_dataset_prepare.sh 2>&1 | tail -60

remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 7 (delta 6), reused 7 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 2.40 KiB | 1.20 MiB/s, done.
From https://github.com/yu37330/Physical_ai
   addb723..ffbee22  fix/colab-dlimp-tensorflow-pin -> origin/fix/colab-dlimp-tensorflow-pin
Updating addb723..ffbee22
Fast-forward
 src/data/convert_selected_lerobot_to_rlds.py | 35 ++++++++++++++++++++++++++--
 tests/test_synthetic_rlds_e2e.py             | 31 ++++++++++++++++++++++++
 2 files changed, 64 insertions(+), 2 deletions(-)
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py", line 114, in _inner_fn
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/_snapshot_download.py", line 332, in snapshot_download
    thread_map(
  File "/usr/local/lib/python3.12/dist-packages/

In [1]:
import getpass, os
os.environ["HF_TOKEN"] = getpass.getpass("HF token: ")

In [3]:
!cd /content/Physical_ai && git pull

/bin/bash: line 1: cd: /content/Physical_ai: No such file or directory


In [4]:
!rm -rf /content/work/rlds/full/parc_libero_plus_selected/incomplete.*
!df -h /content | tail -1

overlay         108G   20G   88G  19% /


In [ ]:
!cd /content/Physical_ai && DATASET_PROFILE=full MANIFEST_FILE=/content/drive/MyDrive/PARC2026/40_experiments/datasets/dataset_manifest.json bash training/openvla_oft_a100/scripts/colab_dataset_prepare.sh 2>&1 | tee /content/work/full.log | grep -v "^\[av1" | tail -3